# Exercice 2 - Entraînez un agent avec une table de décision (Q-table)

On implémente le **Q-Learning** de zéro sur l'environnement **FrozenLake-v1**.

L'idée : construire une table (la Q-table) qui associe à chaque paire (état, action) une valeur représentant la qualité de ce choix. Plus la valeur est haute, meilleure est l'action dans cet état.

**Pourquoi FrozenLake et pas CartPole ?**  
CartPole a un espace d'observation continu (des valeurs décimales infinies), ce qui rend une Q-table inutilisable — elle aurait un nombre infini de lignes.  
FrozenLake a seulement **16 états** discrets (les 16 cases d'une grille 4×4), ce qui est parfait pour une table.


## Étape 1 : Créer l'environnement et initialiser la Q-table

On crée FrozenLake avec `is_slippery=False` pour désactiver le sol glissant. Avec `is_slippery=True`, une action "aller à droite" peut en réalité faire glisser l'agent dans n'importe quelle direction — trop d'aléatoire pour apprendre proprement au début.

La Q-table est un tableau NumPy de forme `(n_états, n_actions)` initialisé à zéro : l'agent ne sait rien au départ.


In [1]:
import gymnasium as gym
import numpy as np


In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False)

n_states = env.observation_space.n  # 16 cases
n_actions = env.action_space.n  # 4 directions : gauche, bas, droite, haut

print(f"Nombre d'états   : {n_states}")
print(f"Nombre d'actions : {n_actions}")

q_table = np.zeros((n_states, n_actions))

print(f"\nDimensions de la Q-table : {q_table.shape}")
print(q_table)

Nombre d'états   : 16
Nombre d'actions : 4

Dimensions de la Q-table : (16, 4)
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


## Étape 2 : Entraîner l'agent avec le Q-Learning

À chaque pas, l'agent choisit entre **explorer** (action aléatoire) ou **exploiter** (meilleure action connue). C'est la stratégie **epsilon-greedy** : epsilon contrôle la probabilité d'exploration. On commence à 1.0 (100% aléatoire) et on le réduit progressivement — l'agent explore beaucoup au début, puis se fie de plus en plus à ce qu'il a appris.

La Q-table est mise à jour avec la **formule de Bellman** :

```
Q(état, action) = Q(état, action) + lr × (récompense + gamma × max(Q(état_suivant)) - Q(état, action))
```

- `lr` (learning rate) : à quelle vitesse l'agent intègre les nouvelles informations
- `gamma` (discount factor) : à quel point l'agent valorise les récompenses futures vs immédiates
- `epsilon` : décroît de façon exponentielle (multiplié par 0.999 à chaque épisode), ce qui est plus progressif qu'une décroissance linéaire


In [3]:
NB_EPISODES = 10000
LEARNING_RATE = 0.8
GAMMA = 0.95
EPSILON = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.999  # facteur de décroissance exponentielle

In [ ]:
epsilon = EPSILON

for episode in range(NB_EPISODES):
    state, info = env.reset()
    terminated = False
    truncated = False

    while not terminated and not truncated:
        # Epsilon-greedy : explorer si le tirage aléatoire est sous epsilon, sinon exploiter  # noqa: E501
        if np.random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state, :])

        new_state, reward, terminated, truncated, info = env.step(action)

        # Mise à jour de Bellman
        future_max_q = np.max(q_table[new_state, :])
        q_table[state, action] = q_table[state, action] + LEARNING_RATE * (
            reward + GAMMA * future_max_q - q_table[state, action]
        )

        state = new_state

    epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)

print("Entraînement terminé.")
print(f"Epsilon final : {epsilon:.3f}")
print("\nQ-table apprise :")
print(np.round(q_table, 2))

Entraînement terminé.
Epsilon final : 0.010

Q-table apprise :
[[0.74 0.77 0.77 0.74]
 [0.74 0.   0.81 0.77]
 [0.77 0.86 0.77 0.81]
 [0.81 0.   0.77 0.77]
 [0.77 0.81 0.   0.74]
 [0.   0.   0.   0.  ]
 [0.   0.9  0.   0.81]
 [0.   0.   0.   0.  ]
 [0.81 0.   0.86 0.77]
 [0.81 0.9  0.9  0.  ]
 [0.86 0.95 0.   0.86]
 [0.   0.   0.   0.  ]
 [0.   0.   0.   0.  ]
 [0.   0.9  0.95 0.86]
 [0.9  0.95 1.   0.9 ]
 [0.   0.   0.   0.  ]]


## Étape 3 : Évaluer les performances de l'agent

On teste l'agent entraîné sur 100 épisodes. Cette fois, **pas d'exploration** : l'agent choisit toujours la meilleure action connue. Dans FrozenLake, une récompense de `1.0` signifie que l'agent a atteint la case objectif (G), `0.0` qu'il est tombé dans un trou.


In [ ]:
NB_EPISODES_EVAL = 100
total_wins = 0

for episode in range(NB_EPISODES_EVAL):
    state, info = env.reset()
    terminated = False
    truncated = False

    while not terminated and not truncated:
        action = np.argmax(
            q_table[state, :]
        )  # toujours la meilleure action, pas de hasard
        state, reward, terminated, truncated, info = env.step(action)

    if reward == 1.0:
        total_wins += 1

taux_reussite = (total_wins / NB_EPISODES_EVAL) * 100
print(f"Taux de réussite sur {NB_EPISODES_EVAL} épisodes : {taux_reussite:.0f}%")

env.close()

Taux de réussite sur 100 épisodes : 100%
